# 📈 Analyse de la vitesse moyenne en Formule 1 par année

Ce notebook analyse l'évolution de la vitesse moyenne des vainqueurs de Grands Prix depuis les années 1950.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src import import_data as i_d

In [ ]:
# Chargement des données depuis le dossier local
# Cette fonction remplit des variables globales comme `results`, `races`, etc.
i_d.charger_donnees_depuis_bureau()

## 📌 Fonction : Calcul de la vitesse moyenne par année

In [ ]:
def compute_speed_per_year(results: pd.DataFrame, races: pd.DataFrame) -> pd.DataFrame:
    """
    Calcule la vitesse moyenne des vainqueurs de F1 par année.

    Parameters
    ----------
    results : pd.DataFrame
        Résultats des pilotes.
    races : pd.DataFrame
        Informations sur les courses.

    Returns
    -------
    pd.DataFrame
        Moyenne annuelle des vitesses (km/h).
    """
    winners = results[results['positionOrder'] == 1]
    merged = winners.merge(races, on='raceId')
    merged['milliseconds'] = pd.to_numeric(merged['milliseconds'], errors='coerce')
    merged = merged.dropna(subset=['milliseconds'])
    merged['hours'] = merged['milliseconds'] / (1000 * 60 * 60)
    merged['speed_kmh'] = 305 / merged['hours']
    merged = merged[merged['speed_kmh'] <= 350]
    return merged.groupby('year')['speed_kmh'].mean().reset_index()

## 📊 Affichage avec régression linéaire améliorée

In [ ]:
def plot_speed_evolution_improved(df_speed: pd.DataFrame) -> None:
    """
    Affiche l'évolution avec régression linéaire + intervalle de confiance.

    Parameters
    ----------
    df_speed : pd.DataFrame
        Contient les colonnes 'year' et 'speed_kmh'.

    Returns
    -------
    None
    """
    sns.set(style="whitegrid")
    plt.figure(figsize=(14, 7))
    x = df_speed['year'].values
    y = df_speed['speed_kmh'].values
    coef = np.polyfit(x, y, 1)
    trend = np.poly1d(coef)
    y_pred = trend(x)
    residuals = y - y_pred
    std_error = np.std(residuals)
    sns.lineplot(x=x, y=y, label="Vitesse moyenne", marker="o")
    plt.plot(x, y_pred, label="Régression (linéaire)", color="red", linewidth=2)
    plt.fill_between(x, y_pred - std_error, y_pred + std_error, color="red", alpha=0.2, label="Intervalle de confiance")
    plt.title("Évolution de la vitesse moyenne des vainqueurs de F1", fontsize=16)
    plt.xlabel("Année")
    plt.ylabel("Vitesse (km/h)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

## ⚙️ Application du traitement

In [ ]:
# Calcul de la vitesse moyenne
speed_df = compute_speed_per_year(results, races)

# Visualisation
plot_speed_evolution_improved(speed_df)